# NVIDIA Dynamo Qwen3-235B-A22B-FP8: a Model That Does Not Fit on One Node

This notebook serves [Qwen/Qwen3-235B-A22B-FP8](https://huggingface.co/Qwen/Qwen3-235B-A22B-FP8)
with [NVIDIA Dynamo](https://github.com/ai-dynamo/dynamo) across two `g6e.12xlarge` nodes, using
[`dgd-multinode-tp4-pp2.yaml`](dgd-multinode-tp4-pp2.yaml).

**This is the case multi-node exists for, and it is a different exercise from the Qwen3-32B
notebook.** There, multi-node was a measurement: the model fits on one node, and spanning two was
a way to price the node boundary. Here there is no single-node arm to compare against, because
there cannot be one:

| | Weights | One `g6e.12xlarge` (4x L40S) | Two |
| --- | --- | --- | --- |
| Qwen3-32B | bf16, ~65 GB | fits with room to spare | fits |
| Qwen3-235B-A22B | **FP8, 239 GB** | **does not fit at any `--gpu-memory-utilization`** | fits |
| Qwen3-235B-A22B | bf16, ~470 GB | no | no — needs 3 nodes, or 2x `g6e.48xlarge` |

One node has 4 x 44.7 GiB = 179 GiB of VRAM against 223 GiB of weights. Two nodes give 358 GiB.
The second node is load-bearing, not an experiment.

**Why not a 70B model?** That is the trap. Llama-3.1-70B in bf16 is ~140 GB and still fits one
4-GPU node, so the obvious "bigger model" tests nothing new. FP8 at 235B is the smallest widely
available open-weight model that *cannot* be served on one node of this instance family.

**Why the eight ranks are `TP=4` x `PP=2` and not `TP=8`.** Not a performance choice — `TP=8`
is arithmetically impossible for this checkpoint. Step 3 works through it; it is the most
transferable thing in this notebook, because the error message names neither tensor parallelism
nor the model.

Read [`../qwen3-32b/README.md`](../qwen3-32b/README.md) first — it covers the six
prerequisites, how the operator turns `node_count: 2` into a LeaderWorkerSet, and why
`--disable-custom-all-reduce` is mandatory on g6e. None of that is repeated here.

## Setup and Imports

In [ ]:
! pip install kubernetes
! pip install openai

In [ ]:
import os
import subprocess
import sys

# Set working directory
os.chdir(os.path.expanduser('~/amazon-eks-machine-learning-with-terraform-and-kubeflow'))
print(f"Working directory: {os.getcwd()}")

# Get the src directory
src_dir = os.path.join(os.getcwd(), "src")
sys.path.insert(0, src_dir)

from k8s.utils import (
    wait_for_helm_release_pods,
    wait_for_dynamo_ready,
    find_k8s_service
)

# Get notebook directory
notebook_dir = os.path.join(os.getcwd(), 'examples', 'inference', 'dynamo', 'vllm',
                            'qwen3-235b-a22b-fp8')
print(f"Notebook directory: {notebook_dir}")

# initialize key variables
release_name = 'dyn-qwen3-235b'
namespace = 'kubeflow-user-example-com'
hf_model_id = 'Qwen/Qwen3-235B-A22B-FP8'
served_model_name = 'qwen3-235b-a22b-fp8'

## Step 1: Verify the Platform and the EFA Fabric

Same three controllers as the 32B notebook, and the same two-part EFA requirement. **Volcano is a
hard gate here and it fails silently** — without it the `DynamoGraphDeployment` is accepted, the
operator authors no `LeaderWorkerSet`, and no pods are ever created.

Unlike the 32B case there is no single-node fallback, so a missing prerequisite does not mean a
slower deployment: it means no deployment.

In [ ]:
for ns in ['dynamo-system', 'lws-system', 'volcano-system']:
    cmd = ['kubectl', 'get', 'pods', '-n', ns]
    result = subprocess.run(cmd, capture_output=True, text=True)
    print(f"===== {ns} =====")
    print(result.stdout or result.stderr)

In [ ]:
import json

# The cudaefa NodePool must admit g6e.12xlarge AND the EFA device plugin must advertise on it.
# One without the other leaves the pod Pending on a resource nothing provides.
cmd = ['kubectl', 'get', 'nodepool', 'cudaefa', '-o', 'json']
result = subprocess.run(cmd, capture_output=True, text=True)
if result.returncode != 0:
    print("cudaefa NodePool not found:", result.stderr)
else:
    np = json.loads(result.stdout)
    for req in np['spec']['template']['spec']['requirements']:
        if req['key'] == 'node.kubernetes.io/instance-type':
            print("g6e.12xlarge admitted by cudaefa:", 'g6e.12xlarge' in req['values'])

cmd = ['kubectl', 'get', 'nodes', '-l', 'karpenter.sh/nodepool=cudaefa', '-o',
       'custom-columns=NODE:.metadata.name,'
       'ZONE:.metadata.labels.topology\\.kubernetes\\.io/zone,'
       'GPU:.status.allocatable.nvidia\\.com/gpu,EFA:.status.allocatable.vpc\\.amazonaws\\.com/efa']
result = subprocess.run(cmd, capture_output=True, text=True)
print(result.stdout or "No cudaefa nodes yet -- Karpenter provisions them in Step 3.")

## Step 2: Stage 239 GB of Weights

This is the long pole and it needs no GPU, so start it well before you want the deployment. The
weights land at `/fsx/pretrained-models/Qwen/Qwen3-235B-A22B-FP8`, which is the path the values
file reads.

`Qwen3-235B-A22B-FP8` is **not a gated model**, so no Hugging Face token is required — the
`hf-snapshot` chart works unmodified. Two things about it matter at this size and not at 8B:

- **Raise `ebs.storage`.** The job downloads to an EBS scratch volume mounted at `/tmp` and then
  copies to FSx, so it needs more than 239 GB of scratch. The chart's default is `400Gi`, which is
  enough, but it is close enough to the model size that it is worth setting deliberately rather
  than inheriting.
- **It writes everything twice.** Download to EBS, then `shutil.copy2` file-by-file onto Lustre.
  Budget hours, not minutes, and run it detached rather than watching it.

Check free space on FSx first. 239 GB has to fit alongside whatever else is already there.

In [ ]:
# Check FSx capacity before starting. Any pod with the pv-fsx claim will do.
cmd = ['kubectl', 'get', 'pods', '-n', namespace, '-o', 'jsonpath={.items[0].metadata.name}']
pod = subprocess.run(cmd, capture_output=True, text=True).stdout.strip()
if pod:
    cmd = ['kubectl', 'exec', '-n', namespace, pod, '--', 'df', '-h', '/fsx']
    print(subprocess.run(cmd, capture_output=True, text=True).stdout)
else:
    print("No pod with /fsx mounted yet -- check capacity after Step 3 starts.")

In [ ]:
cmd = [
    'helm', 'install', '--debug', f'{release_name}-weights',
    'charts/machine-learning/model-prep/hf-snapshot',
    '--set-json', f'env=[{{"name":"HF_MODEL_ID","value":"{hf_model_id}"}}]',
    '--set', 'ebs.storage=500Gi',
    '-n', namespace
]

result = subprocess.run(cmd, capture_output=True, text=True)
print(result.stdout)
if result.stderr:
    print("STDERR:", result.stderr)

In [ ]:
# 239 GB downloaded to EBS and then copied to FSx. This is hours, not minutes.
wait_for_helm_release_pods(release_name=f'{release_name}-weights', namespace=namespace,
                           timeout=36000)

In [ ]:
# Check against the index, not a directory listing: it names every shard the engine will open, so
# it catches a partial download.
check = '''
import json, os
d = "/fsx/pretrained-models/Qwen/Qwen3-235B-A22B-FP8"
m = json.load(open(os.path.join(d, "model.safetensors.index.json")))
shards = sorted(set(m["weight_map"].values()))
missing = [s for s in shards if not os.path.exists(os.path.join(d, s))]
total = sum(os.path.getsize(os.path.join(d, s)) for s in shards if not missing)
print(f"index names {len(shards)} shards; missing: {missing or 'none'}")
print(f"index total_size {m['metadata']['total_size']/1e9:.1f} GB, on disk {total/1e9:.1f} GB")
'''
cmd = ['kubectl', 'get', 'pods', '-n', namespace, '-o', 'jsonpath={.items[0].metadata.name}']
pod = subprocess.run(cmd, capture_output=True, text=True).stdout.strip()
cmd = ['kubectl', 'exec', '-n', namespace, pod, '--', 'python3', '-c', check]
result = subprocess.run(cmd, capture_output=True, text=True)
print(result.stdout or result.stderr)

In [ ]:
# Uninstall the Helm chart after completion
cmd = ['helm', 'uninstall', f'{release_name}-weights', '-n', namespace]
result = subprocess.run(cmd, capture_output=True, text=True)
print(result.stdout)
if result.stderr:
    print("STDERR:", result.stderr)

### An alternative worth knowing: stage through S3 instead

If your FSx filesystem has a data repository association with
`auto_import_policy` covering `NEW` — which is what `main.tf` configures — then anything written
to the association's S3 prefix appears under `/fsx` on its own, with no import task and no
in-cluster job at all. Downloading on a host and running `aws s3 sync` into
`<import_path>/pretrained-models/Qwen/Qwen3-235B-A22B-FP8/` is often faster than the chart,
because it skips the EBS round-trip entirely.

It also keeps any Hugging Face token out of the cluster, which matters for *gated* models. This
one is not gated, so the `hf-snapshot` path above is the simpler choice here.

Either way, be aware that **FSx imports file data lazily**. Auto-import creates the metadata
immediately — the files appear in `ls` at full size — but the first process to read a byte pays
for the fetch from S3. On a `SCRATCH_2` filesystem that fetch is throughput-limited, so for 239 GB
the first engine launch can spend a long time apparently doing nothing. Reading the shards once
from any pod with `/fsx` mounted moves that cost off the critical path:

    find /fsx/pretrained-models/Qwen/Qwen3-235B-A22B-FP8 -name '*.safetensors' \
      | xargs -P 8 -I{} dd if={} of=/dev/null bs=16M iflag=direct status=none

## Step 3: Launch TP=4 x PP=2 Across Two Nodes

Nothing in [`dgd-multinode-tp4-pp2.yaml`](dgd-multinode-tp4-pp2.yaml) writes a `LeaderWorkerSet`.
`node_count: 2` is the entire multi-node input; the operator authors the LWS, and LWS injects
`LWS_LEADER_ADDRESS`, `LWS_WORKER_INDEX` and `LWS_GROUP_SIZE` into every pod in the group.

### Why not TP=8

The obvious geometry for 8 GPUs is `TP=8`, and it is the one the 32B notebook uses on exactly this
hardware. It cannot work here. It fails a few minutes into startup, well after distributed init
has succeeded and `world_size=8` has been logged, with an error that mentions neither tensor
parallelism nor this model:

```
ValueError: The output_size of gate's and up's weight = 192 is not divisible by
            weight quantization block_n = 128.
```

Two numbers in `config.json` produce that:

| | |
| --- | --- |
| `quantization_config.weight_block_size` | `[128, 128]` — FP8 scales are per 128x128 block, not per tensor |
| `moe_intermediate_size` | `1536` |

Tensor parallelism splits each expert's intermediate dimension across the ranks, so a rank's shard
is `1536 / TP` — and that shard has to be a whole number of 128-wide quantization blocks. Since
`1536 / 128 = 12`, **`TP` must divide 12**:

| `TP` | shard | blocks | legal |
| --- | --- | --- | --- |
| 2 | 768 | 6 | yes |
| 3 | 512 | 4 | yes |
| 4 | 384 | 3 | yes |
| 6 | 256 | 2 | yes |
| **8** | **192** | **1.5** | **no** |
| 12 | 128 | 1 | yes |

The constraint belongs to the checkpoint, not to the GPUs, the fabric or vLLM's version — no
flag relaxes it, and a 4-GPU-per-node instance type cannot be rearranged around it. `TP=4` is the
largest legal width that divides a node evenly, and all 8 GPUs are needed to hold the weights, so
the second node has to be reached by **pipeline** rather than tensor parallelism.

Worth checking before you pick a parallel width for any block-quantized MoE: take
`moe_intermediate_size / block_n` and factor it. That is the complete list of legal `TP` values.

### What the arrangement buys, and what it costs

Being forced into `PP=2` is not a penalty here. The pipeline boundary lands exactly on the node
boundary, so the only cross-node traffic is activations handed between stages once per microbatch,
rather than a TP all-reduce in every one of the 94 layers. The cost is the usual pipeline one:
stages idle unless enough requests are in flight to keep them fed, trading single-request latency
for throughput. [`../qwen3-32b/serve.ipynb`](../qwen3-32b/serve.ipynb) measures that trade on a
model small enough to also run `TP=8`.

Two further notes on the file:

- **`TP x PP = 8` is a floor, not a tuning choice.** It is what makes the weights fit, and
  separately it is what makes the operator inject the multi-node launch path: that injection is
  conditional on `TP x PP` exceeding the container's GPU request. At a product of 4 with
  `nvidia.com/gpu: 4` the operator would build a single-node engine on the leader — which for
  this model means an out-of-memory crash rather than a merely wasteful deployment.
- **`--max-model-len 16384` is set for comparability, not forced by memory.** The arithmetic is in
  Step 4, and it is more forgiving than it first looks.

Per-rank budget:

| | |
| --- | --- |
| weights per rank | 239 GB / 8 = 29.9 GB = **27.8 GiB** |
| L40S usable | 44.7 GiB |
| budget at `--gpu-memory-utilization 0.85` | 38.0 GiB |
| left for KV cache, CUDA graphs, NIXL buffers | **~10 GiB** |

That is a far thinner margin than the 32B's ~30 GiB of slack, which is why `0.85` is not raised
here. Pushing past ~0.90 on a card this full tends to fail during CUDA graph capture, which is a
more confusing failure than an honest out-of-memory at load.

### If you want a TP=8-shaped arm anyway

`--enable-expert-parallel` is the one flag that would make one legal: it assigns whole experts to
ranks instead of slicing each expert's intermediate dimension, so nothing is divided by `TP` and
the rule above stops applying. That is a different memory layout and a different traffic pattern
(all-to-all in place of the MoE all-reduce), so it is a second arm to measure, not a drop-in
substitute — see the note on that flag in the values file.

In [ ]:
cmd = [
    'helm', 'install', '--debug', release_name,
    'charts/machine-learning/serving/dynamo/',
    '-f', f'{notebook_dir}/dgd-multinode-tp4-pp2.yaml',
    '-n', namespace
]

result = subprocess.run(cmd, capture_output=True, text=True)
print(result.stdout)
if result.stderr:
    print("STDERR:", result.stderr)

### While it loads: what "slow" and "hung" look like

Two GPU nodes have to be provisioned, then all 8 ranks have to read their share of 239 GB off
FSx — and on a cold filesystem that read comes from S3. Expect this to take substantially longer
than the 32B equivalent, and give the wait a generous timeout.

Before concluding it is stuck, distinguish the two causes, because they look identical from
outside:

| | MNNVL deadlock | slow cold FSx read |
| --- | --- | --- |
| `Loading safetensors checkpoint shards` line | never appears | appears, advances slowly |
| `rchar` in `/proc/<rank-pid>/io`, sampled 30s apart | flat | climbing at tens of MB/s |
| `nvidia-smi` on the leader | 0%, ~800 MiB | 0%, climbing |
| `py-spy dump --pid <rank>` | `rendezvous` / `_init_mnnvl_buffer` | a weight-loading frame |

The deadlock is what `--disable-custom-all-reduce` prevents; if the values file still carries it,
the right-hand column is the one you are in.

In [ ]:
wait_for_dynamo_ready(release_name=release_name, namespace=namespace, timeout=7200)

In [ ]:
import time

# Run this in a separate cell while the wait above is in flight if you want to watch progress.
cmd = ['kubectl', 'get', 'pods', '-n', namespace,
       '-l', 'nvidia.com/dynamo-component-type=worker',
       '-o', 'jsonpath={.items[0].metadata.name}']
leader = subprocess.run(cmd, capture_output=True, text=True).stdout.strip()

def total_rchar(pod):
    """Sum bytes read by every process in the pod. Climbing means weights are loading."""
    script = ('for p in /proc/[0-9]*/io; do grep -H ^rchar $p 2>/dev/null; done '
              '| awk -F: \'{s+=$3} END {print s}\'')
    out = subprocess.run(['kubectl', 'exec', '-n', namespace, pod, '--', 'bash', '-c', script],
                         capture_output=True, text=True).stdout.strip()
    return int(out) if out.isdigit() else 0

if leader:
    a = total_rchar(leader)
    time.sleep(30)
    b = total_rchar(leader)
    print(f"{leader}: rchar {a/1e9:.1f} GB -> {b/1e9:.1f} GB over 30s "
          f"({(b - a)/30/1e6:.0f} MB/s)")
    print("Each node reads roughly half of 239 GB. Flat, with no safetensors line in the log, "
          "means a deadlock rather than a slow read.")

## Step 4: Confirm It Is Really One Engine Across Two Nodes

For this model the check is less about catching a silent misconfiguration — a `TP=4` engine would
have crashed rather than quietly wasted a node — and more about confirming the fabric is live.
A group split across availability zones falls back to TCP with no error anywhere, and at this
model size a tensor-parallel all-reduce per layer over TCP is the difference between usable and
not.

In [ ]:
cmd = ['kubectl', 'get', 'lws', '-n', namespace, '-o', 'wide']
result = subprocess.run(cmd, capture_output=True, text=True)
print("===== LeaderWorkerSet =====")
print(result.stdout or "NONE -- the operator did not author one. Check Volcano is installed.")

cmd = ['kubectl', 'get', 'pods', '-n', namespace,
       '-l', 'nvidia.com/dynamo-component-type=worker', '-o',
       'custom-columns=POD:.metadata.name,'
       'IDX:.metadata.labels.leaderworkerset\\.sigs\\.k8s\\.io/worker-index,'
       'NODE:.spec.nodeName,GPU:.spec.containers[0].resources.requests.nvidia\\.com/gpu,'
       'EFA:.spec.containers[0].resources.requests.vpc\\.amazonaws\\.com/efa']
result = subprocess.run(cmd, capture_output=True, text=True)
print("===== worker pods =====")
print(result.stdout)

# Both pods must be in the SAME zone: EFA traffic is not routed between subnets.
cmd = ['kubectl', 'get', 'pods', '-n', namespace,
       '-l', 'nvidia.com/dynamo-component-type=worker',
       '-o', 'jsonpath={range .items[*]}{.spec.nodeName}{"\n"}{end}']
nodes = subprocess.run(cmd, capture_output=True, text=True).stdout.split()
zones = set()
for node in nodes:
    cmd = ['kubectl', 'get', 'node', node, '-o',
           'jsonpath={.metadata.labels.topology\\.kubernetes\\.io/zone}']
    zones.add(subprocess.run(cmd, capture_output=True, text=True).stdout.strip())
print(f"===== placement =====\n{len(set(nodes))} distinct nodes, zone(s): {zones}")
if len(zones) > 1:
    print("WARNING: group is split across zones. EFA is not routed between subnets -- this "
          "deployment is silently running over TCP.")

In [ ]:
cmd = ['kubectl', 'logs', '-n', namespace, '-l', 'nvidia.com/dynamo-component-type=worker',
       '--tail=6000', '--prefix']
result = subprocess.run(cmd, capture_output=True, text=True)

interesting = ('world_size', 'distributed_executor_backend', 'Loading safetensors',
               'Available KV cache memory', 'GPU KV cache size', 'graph capturing',
               'MNNVL', 'Using network', 'quantization', 'fp8', 'num_experts')
for line in result.stdout.splitlines():
    if any(k in line for k in interesting):
        print(line)

### What to read in that output

`GPU KV cache size` is the number worth pausing on, because two things about this arrangement
shape it.

**The attention sharding comes out even.** Qwen3-235B-A22B has 4 key/value heads, and `TP=4` gives
each rank exactly one, with no replication. This is a second reason not to reach for `TP=8` even
where the quantization blocks allow it: against 8 ranks vLLM does not refuse to shard 4 heads, it
replicates each head across two ranks, so the cache ends up stored **twice** cluster-wide and
aggregate KV capacity stops growing once rank count passes the KV head count.

**Pipeline parallelism halves the per-rank layer count.** Each rank holds 47 of the 94 layers, so
per-rank KV is `2 x 47 layers x 1 head x 128 head_dim x 2 bytes` = **23 KiB per token**.

The practical consequence is that `--max-model-len` is far less constrained here than the tight
weights-per-rank figure suggests: ~10 GiB of headroom at 23 KiB/token is several hundred thousand
tokens of cache per rank. The binding limit is the model's own `max_position_embeddings` of 40960,
not memory. Budget per rank, never per cluster.

Also note the rank naming. The workers log as `Worker_PP0_TP0 ... Worker_PP0_TP3` on the leader
and `Worker_PP1_TP0 ... Worker_PP1_TP3` on the second pod. That is the quickest confirmation that
the pipeline boundary fell on the node boundary rather than cutting across it — a single
`PP` index per pod is what you want to see.

If the engine did die with `CUDA out of memory`, lower `--max-model-len` first. Raising
`--gpu-memory-utilization` is the wrong lever on a card this full.

### What this run actually produced

| | |
| --- | --- |
| `Loading weights took` | **1298 s** (21m38s), 48/48 shards at ~27 s each |
| `init engine (profile, create kv cache, warmup model) took` | 72.6 s, of which 42.9 s compilation |
| `GPU KV cache size` | **307,312 tokens** — `Maximum concurrency for 16,384 tokens per request: 18.76x` |

Two things to take from that. First, 307k tokens against a ~10 GiB budget is consistent with the
23 KiB/token figure above, so the arithmetic holds; `--max-model-len 16384` is nowhere near the
memory ceiling, and the 18.76x concurrency headroom is real.

Second, **the 21-minute load is storage-bound, not network-bound, and it is the dominant cost of
every launch.** vLLM's loader is per-rank: all 8 ranks open the shards independently, so measured
per-rank read throughput was ~36 MiB/s and ~290 MiB/s in aggregate, which is what a 1.2 TiB
`SCRATCH_2` FSx for Lustre filesystem delivers. The filesystem is saturated; the 100 Gbps fabric is
nearly idle. If you care about launch time, the lever is FSx throughput (a larger filesystem, or
`PERSISTENT` with provisioned throughput per TiB), not the interconnect. Pre-warming from S3 before
launch removes the lazy-fetch penalty but not this, because this is the filesystem's own ceiling.

There is also a useful incidental confirmation of the divisibility argument in the engine's own
logs. vLLM looks for a tuned MoE kernel config and names the shape it needs:

```
WARNING fused_moe.py:1161] Using default MoE config. Performance might be sub-optimal!
  Config file not found at .../configs/E=128,N=384,device_name=NVIDIA_L40S,
  dtype=fp8_w8a8,block_shape=[128,128].json
```

`E=128` is the expert count and **`N=384` is the per-rank expert intermediate shard** — exactly
`1536 / 4`, three whole 128-wide blocks. At `TP=8` that number would have been 192, and `N=192`
with `block_shape=[128,128]` is the shape that cannot exist. (The warning itself is benign: no
tuned config ships for this shape on L40S, so the generic kernel is used.)

## Step 5: Test the Service

    kubectl port-forward -n kubeflow-user-example-com svc/dyn-qwen3-235b-frontend 8000:8000

The API surface is identical to the 8B and 32B examples. That is the point worth taking away: a
model that physically cannot fit on one machine is, to the client, the same OpenAI-compatible
endpoint.

In [ ]:
import time
from openai import OpenAI, APIConnectionError

client = OpenAI(base_url="http://localhost:8000/v1", api_key="dummy")

def list_models(client, attempts=30, interval=10):
    """List served models, waiting for the port-forward rather than failing on a refused
    connection. A `kubectl port-forward` is bound to one pod, so it dies whenever the frontend
    pod is replaced -- which makes a dead tunnel look exactly like a broken deployment."""
    for attempt in range(attempts):
        try:
            return [model.id for model in client.models.list()]
        except APIConnectionError:
            print(f"No answer on localhost:8000 (attempt {attempt + 1}/{attempts}). "
                  f"Is the port-forward running?")
            time.sleep(interval)
    raise RuntimeError("Gave up waiting for the frontend on localhost:8000")

print(list_models(client))

In [ ]:
response = client.chat.completions.create(
    model=served_model_name,
    messages=[{"role": "user", "content": "Explain the concept of attention masking in causal language models and its importance for autoregressive generation."}],
    max_tokens=2048
)

print(response.choices[0].message.content)

Qwen3 is a reasoning model and emits a `<think>` block before its answer, so keep `max_tokens`
generous. At 128 the response is truncated mid-reasoning and looks like a broken deployment rather
than a budget you set too low.

A22B means **22 billion active parameters** out of 235 billion: 8 of 128 experts fire per token.
So decode touches far less weight per token than the parameter count suggests, while the *memory*
footprint is set by all 235B. That asymmetry is the whole economic case for MoE, and it is why
this model is servable on L40S at all.

## Step 6: Where Expert Parallelism Comes In

The values file leaves MoE routing on the default, which applies tensor parallelism to the
experts — each expert's intermediate dimension is sharded across all 8 ranks. The alternative is
`--enable-expert-parallel`, which assigns whole experts to ranks and replaces the MoE all-reduce
with an all-to-all.

On a 100 Gbps fabric with 16 experts per rank that trade is worth measuring, but it is a **second
arm, not a better default**. Expert parallelism makes per-rank memory depend on how many experts
land on each rank, so it changes the sizing argument in Step 3 as well as the traffic pattern — you
cannot swap the flag in and reuse the same headroom calculation.

If you run it, run it as its own install with its own values file, and compare against this one on
the same workload.

## Step 7: Stop the Service

Check `consolidationPolicy` before assuming the GPU nodes go away. With `consolidateAfter: Never`
the empty nodes stay — which is what you want between arms of an experiment, and not what you want
when you are finished.

Re-acquiring two `g6e.12xlarge` in a single availability zone is not guaranteed, so if you intend
to run another arm, keep the nodes rather than reclaiming and re-requesting them.

In [ ]:
cmd = ['helm', 'uninstall', release_name, '-n', namespace]
result = subprocess.run(cmd, capture_output=True, text=True)
print(result.stdout)
if result.stderr:
    print("STDERR:", result.stderr)

cmd = ['kubectl', 'get', 'nodepool', '-o',
       'custom-columns=NAME:.metadata.name,POLICY:.spec.disruption.consolidationPolicy,'
       'AFTER:.spec.disruption.consolidateAfter']
result = subprocess.run(cmd, capture_output=True, text=True)
print(result.stdout)

## Where to Go Next

- [`../qwen3-32b/README.md`](../qwen3-32b/README.md) — the prerequisites, the LeaderWorkerSet
  mechanism in full, and what a node boundary costs when the model *does* fit on one node.
- [`../qwen3-32b/serve.ipynb`](../qwen3-32b/serve.ipynb) — `TP=8` vs `TP=4`+`PP=2` vs two
  replicas behind a KV-aware router, all on the same 8 GPUs.
- [`../qwen3-8b/bench/README.md`](../qwen3-8b/bench/README.md) — the load generator and the
  method for comparing arms, including the two instruments needed to tell a working KV router from
  one that is buying nothing.